In [6]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict

In [20]:
class BatsmanState(TypedDict):

    runs : int
    balls : int
    fours : int
    sixes : int
    sr : float #strike rate
    bpb : float #balls per boundary
    boundary_percent: float
    summary  : str

In [25]:
def calculate_sr(state: BatsmanState):

    sr = (state['runs']/state['balls'])*100
    # state['sr'] = sr
    return {'sr':sr}#partial update is done during parallel workflows

def calculate_bpb(state: BatsmanState):

     # Safeguard against zero division if no boundaries are hit
    denom = state['fours'] + state['sixes']
    bpb = state['balls'] / denom if denom > 0 else 0
    return {"bpb": bpb}

def calculate_boundary_percent(state: BatsmanState):

    boundary_percent = (((state['fours'] * 4) + (state['sixes'] * 6))/state['runs'])*100
    return {'boundary_percent': boundary_percent}

def summary(state: BatsmanState):

    summary = f"""
Strike Rate - {state['sr']} \n
Balls per boundary - {state['bpb']} \n
Boundary percent - {state['boundary_percent']}
"""
      
    return {'summary': summary}



In [26]:
graph = StateGraph(BatsmanState)

graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node('summary', summary)

# edges

graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_bpb')
graph.add_edge(START, 'calculate_boundary_percent')

graph.add_edge('calculate_sr', 'summary')
graph.add_edge('calculate_bpb', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')

graph.add_edge('summary', END)

# graph.compile()
workflow = graph.compile()

In [27]:
intial_state = {
    'runs': 100,
    'balls': 50,
    'fours': 6,
    'sixes': 4
}

final_result = workflow.invoke(intial_state)
print(final_result['summary'])



Strike Rate - 200.0 

Balls per boundary - 5.0 

Boundary percent - 48.0

